# Experiment 10: Fine-Tuning a Pretrained Transformer (RoBERTa) for Resume Classification
**Mini-project:** Resume Analyzer Using Natural Language Processing (Group 28)

**Objective:** Fine-tune a pretrained large language model (RoBERTa-base, Hugging Face Transformers) to classify resumes into five domains and compare it with a TF-IDF baseline using F1-score.

Open in Colab: [Exp10_Fine_Tuning_RoBERTa.ipynb](https://colab.research.google.com/github/GnsShubham06/resume-analyzer-nlp/blob/main/notebooks/experiments/Exp10_Fine_Tuning_RoBERTa.ipynb) &nbsp;|&nbsp; Run with **Runtime → Run all**.
Data: 100 resumes (20 per domain); the screenshots in the report use five fictional (synthetic) resumes.

> **Tip:** in Colab choose *Runtime → Change runtime type → T4 GPU*; fine-tuning then takes about 3 minutes (about 20 minutes on a CPU).

In [ ]:
# ---- Setup: runs in Google Colab and locally --------------------------------
# In Colab this cell downloads the project repository (code + anonymised
# dataset) from GitHub and installs the packages Colab does not ship with.
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
REPO = "https://github.com/GnsShubham06/resume-analyzer-nlp"
if "google.colab" in sys.modules:
    if not os.path.isdir("/content/resume-analyzer-nlp"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "/content/resume-analyzer-nlp"], check=True)
    os.chdir("/content/resume-analyzer-nlp")
    extra = []
    if extra:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra], check=True)
else:  # local run from notebooks/experiments/
    for _ in range(3):
        if os.path.isfile("requirements.txt"):
            break
        os.chdir("..")
sys.path.insert(0, os.getcwd())

import nltk
for pkg in []:
    nltk.download(pkg, quiet=True)

%config InlineBackend.figure_format = "retina"
import pandas as pd
pd.set_option("display.max_colwidth", 120)
df = pd.read_csv("data/resume_dataset.csv")
df["extracted_text"] = df["extracted_text"].fillna("")
print("Project folder:", os.path.basename(os.getcwd()), "| resumes:", len(df))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 9, "axes.titlesize": 10})
ORDER = ["Engineering", "HR", "Finance", "Healthcare", "Agriculture"]
SAMPLE_IDS = {"Engineering": "ENG_06", "HR": "HR_06", "Finance": "FIN_07",
              "Healthcare": "HLT_04", "Agriculture": "AGR_02"}     # five fictional resumes
splits = pd.read_csv("data/processed/splits.csv")[["resume_id", "split"]]
df = df.merge(splits, on="resume_id")

import copy, random, tempfile, time
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
LABEL2ID = {c: i for i, c in enumerate(ORDER)}
train_df, val_df, test_df = (df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
print("train / validation / test resumes:", len(train_df), len(val_df), len(test_df))
print(test_df.groupby(["category", "source_type"]).size().unstack(fill_value=0).loc[ORDER].to_string())

## a. Fine-tuning process

In [ ]:
from matplotlib.patches import FancyBboxPatch
steps = ["100 resumes\n(PDF text)", "clean +\nanonymise", "resume-level split\n60 / 20 / 20",
         "RoBERTa tokenizer\n(512 tokens)", "pretrained RoBERTa-base\n+ new 5-class head",
         "train: cross-entropy,\nAdamW, 6 epochs", "keep best epoch\n(validation macro-F1)", "test once:\nF1 vs TF-IDF"]
fig, ax = plt.subplots(figsize=(8.6, 1.9)); ax.axis("off")
w, gap = 0.112, 0.015
for i, s in enumerate(steps):
    x = i * (w + gap)
    dark = i in (4, 5)
    ax.add_patch(FancyBboxPatch((x, 0.2), w, 0.6, boxstyle="round,pad=0.01", fc="black" if dark else "white",
                                ec="black", lw=1))
    ax.text(x + w / 2, 0.5, s, ha="center", va="center", fontsize=6.6, color="white" if dark else "black")
    if i < len(steps) - 1:
        ax.annotate("", xy=(x + w + gap, 0.5), xytext=(x + w, 0.5), arrowprops=dict(arrowstyle="->", lw=1))
ax.set_xlim(-0.01, len(steps) * (w + gap)); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

## b. Baseline: TF-IDF + logistic regression (same split)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
baseline = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, max_df=0.9),
                         LogisticRegression(max_iter=3000, C=10))
baseline.fit(train_df.classification_text, train_df.category)
base_pred = baseline.predict(test_df.classification_text)
print(f"baseline test accuracy {accuracy_score(test_df.category, base_pred):.2f}, "
      f"macro-F1 {f1_score(test_df.category, base_pred, average='macro'):.2f}")

## c. Fine-tuning RoBERTa-base (explicit PyTorch training loop)

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
MODEL, EPOCHS, LR, BATCH, MAX_LEN, SEED = "roberta-base", 6, 5e-5, 2, 512, 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=5, id2label=dict(enumerate(ORDER)), label2id=LABEL2ID).to(device)
# fingerprint of every pretrained tensor (cheap on memory) to prove later that fine-tuning changed it
fingerprint = lambda t: float(t.detach().double().abs().sum()) + float(t.detach().double().flatten()[::997].sum())
pretrained = {k: fingerprint(v) for k, v in model.named_parameters()}
BEST_PATH = os.path.join(tempfile.gettempdir(), "roberta_resume_best.pt")

def loader(frame, shuffle):
    enc = tokenizer(list(frame.transformer_text), truncation=True, max_length=MAX_LEN, padding="max_length",
                    return_tensors="pt")
    labels = torch.tensor([LABEL2ID[c] for c in frame.category])
    data = list(zip(enc["input_ids"], enc["attention_mask"], labels))
    return DataLoader(data, batch_size=BATCH, shuffle=shuffle, generator=torch.Generator().manual_seed(SEED))
train_dl, val_dl, test_dl = loader(train_df, True), loader(val_df, False), loader(test_df, False)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total = EPOCHS * len(train_dl)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total), total)
loss_fn = torch.nn.CrossEntropyLoss()

def evaluate(dl):
    model.eval(); preds, probs, losses = [], [], []
    with torch.no_grad():
        for ids, mask, y in dl:
            logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            losses.append(loss_fn(logits, y.to(device)).item())
            probs += torch.softmax(logits, -1).cpu().tolist(); preds += logits.argmax(-1).cpu().tolist()
    return np.mean(losses), preds, probs

log, step_losses, best_f1 = [], [], -1
for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); ep_losses = []
    for ids, mask, y in train_dl:
        logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits   # forward pass
        loss = loss_fn(logits, y.to(device))                                               # classification loss
        optimizer.zero_grad(); loss.backward()                                             # backpropagation
        optimizer.step(); scheduler.step()                                                 # parameter update
        ep_losses.append(loss.item()); step_losses.append(loss.item())
    val_loss, vp, _ = evaluate(val_dl)
    vy = [LABEL2ID[c] for c in val_df.category]
    acc, f1 = accuracy_score(vy, vp), f1_score(vy, vp, average="macro")
    saved = f1 > best_f1
    if saved:
        best_f1 = f1
        torch.save(model.state_dict(), BEST_PATH)                     # best checkpoint on disk
    log.append((epoch, np.mean(ep_losses), val_loss, acc, f1, time.time() - t0, "yes" if saved else ""))
    print(f"epoch {epoch}: train loss {np.mean(ep_losses):.4f} | val loss {val_loss:.4f} | val acc {acc:.2f} | "
          f"val macro-F1 {f1:.2f} | {time.time() - t0:.0f}s {'<- best, saved' if saved else ''}")
model.load_state_dict(torch.load(BEST_PATH, map_location=device))

In [ ]:
log_df = pd.DataFrame(log, columns=["epoch", "train loss", "val loss", "val accuracy", "val macro-F1", "seconds",
                                    "best so far"]).round(4)
display(log_df.set_index("epoch"))
fig, ax = plt.subplots(figsize=(6.2, 2.5))
steps_per_epoch = len(train_dl)
ax.plot(np.arange(1, len(step_losses) + 1) / steps_per_epoch, step_losses, color="#aaaaaa", lw=0.8,
        label="training loss (each step)")
ax.plot(log_df.epoch, log_df["train loss"], "k-o", ms=4, label="training loss (epoch mean)")
ax.plot(log_df.epoch, log_df["val loss"], "k--s", ms=4, mfc="white", label="validation loss")
ax.set_xlabel("epoch"); ax.set_ylabel("cross-entropy loss"); ax.legend(fontsize=7)
ax.set_title(f"Fine-tuning RoBERTa-base ({device}, lr {LR}, batch {BATCH})"); plt.tight_layout(); plt.show()

## d. Test set: fine-tuned RoBERTa vs TF-IDF baseline

In [ ]:
_, test_pred_ids, test_probs = evaluate(test_dl)
ft_pred = [ORDER[i] for i in test_pred_ids]
true = test_df.category.tolist()
def metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return accuracy_score(y_true, y_pred), p, r, f
rows = []
for name, pred in [("TF-IDF + Logistic Regression", list(base_pred)), ("Fine-tuned RoBERTa-base", ft_pred)]:
    for subset, m in [("all 20", np.ones(len(true), bool)),
                      ("5 supplied originals", test_df.source_type.eq("supplied_original").to_numpy()),
                      ("15 fictional", test_df.source_type.eq("synthetic").to_numpy())]:
        acc, p, r, f = metrics([t for t, k in zip(true, m) if k], [q for q, k in zip(pred, m) if k])
        rows.append((name, subset, acc, p, r, f))
display(pd.DataFrame(rows, columns=["model", "test resumes", "accuracy", "macro precision", "macro recall",
                                    "macro-F1"]).round(3))
fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.1))
for ax, (name, pred) in zip(axes, [("TF-IDF baseline", base_pred), ("Fine-tuned RoBERTa", ft_pred)]):
    cm = confusion_matrix(true, pred, labels=ORDER)
    ax.imshow(cm, cmap="Greys", vmin=0, vmax=4)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > 2 else "black")
    ax.set_xticks(range(5), [c[:5] for c in ORDER], fontsize=7); ax.set_yticks(range(5), [c[:5] for c in ORDER], fontsize=7)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(name)
plt.tight_layout(); plt.show()

## e. Did fine-tuning really change the pretrained model?

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
changed = [k for k, v in model.named_parameters() if k in pretrained and fingerprint(v) != pretrained[k]]
new_head = [k for k in pretrained if k.startswith("classifier")]
print(f"parameters: {n_params:,} (trainable: {n_train:,})")
print(f"pretrained tensors changed by fine-tuning: {len([k for k in changed if not k.startswith('classifier')])} "
      f"of {len([k for k in pretrained if not k.startswith('classifier')])}; new classification head: {new_head}")
print("BERTScore is a metric for generated text; our output is a class label, so we report accuracy and F1.")
demo = test_df[test_df.source_type == "synthetic"].groupby("category").head(1).set_index("resume_id")
for rid, r in demo.loc[:, ["category"]].iterrows():
    i = test_df.index[test_df.resume_id == rid][0]
    print(f"{rid}: true {r.category:<12} predicted {ft_pred[i]:<12} confidence {max(test_probs[i]):.2f}")